# Government Procurement Price Benchmarking System
## AI Model for Price Prediction

This notebook contains:
1. Synthetic Dataset Generation
2. Data Exploration & Visualization
3. AI Model Training (Random Forest & Gradient Boosting)
4. Model Evaluation
5. Price Prediction Examples

## 1. Install & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import random
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

print("Libraries imported successfully!")

## 2. Dataset Generation

In [ ]:
# Product categories with base prices and variance
PRODUCT_CATEGORIES = {
    "Networking Equipment": {
        "items": [
            {"name": "Enterprise Router", "base_price": 45000, "variance": 0.25},
            {"name": "Managed Switch 24-Port", "base_price": 35000, "variance": 0.20},
            {"name": "Managed Switch 48-Port", "base_price": 75000, "variance": 0.20},
            {"name": "Wireless Access Point", "base_price": 12000, "variance": 0.30},
            {"name": "Network Firewall", "base_price": 150000, "variance": 0.35},
            {"name": "Load Balancer", "base_price": 200000, "variance": 0.30},
        ],
        "vendors": ["Cisco", "HP Enterprise", "Juniper", "Fortinet", "D-Link", "TP-Link"]
    },
    "Computing Hardware": {
        "items": [
            {"name": "Desktop Workstation", "base_price": 85000, "variance": 0.20},
            {"name": "Server Rack Mount", "base_price": 350000, "variance": 0.25},
            {"name": "Laptop Standard", "base_price": 55000, "variance": 0.15},
            {"name": "Laptop High-End", "base_price": 95000, "variance": 0.20},
            {"name": "Graphics Workstation", "base_price": 250000, "variance": 0.30},
            {"name": "Thin Client", "base_price": 18000, "variance": 0.15},
        ],
        "vendors": ["Dell", "HP", "Lenovo", "Acer", "ASUS", "Apple"]
    },
    "Storage Systems": {
        "items": [
            {"name": "Hard Disk Drive 1TB", "base_price": 4500, "variance": 0.15},
            {"name": "Hard Disk Drive 4TB", "base_price": 12000, "variance": 0.20},
            {"name": "SSD 512GB", "base_price": 5500, "variance": 0.25},
            {"name": "SSD 2TB", "base_price": 18000, "variance": 0.25},
            {"name": "NAS Storage 4-Bay", "base_price": 45000, "variance": 0.20},
            {"name": "SAN Storage Array", "base_price": 500000, "variance": 0.30},
        ],
        "vendors": ["Seagate", "Western Digital", "Synology", "QNAP", "Dell EMC", "NetApp"]
    },
    "Display Systems": {
        "items": [
            {"name": "LED Monitor 24-inch", "base_price": 14000, "variance": 0.15},
            {"name": "LED Monitor 27-inch", "base_price": 22000, "variance": 0.20},
            {"name": "4K Monitor 32-inch", "base_price": 38000, "variance": 0.25},
            {"name": "Interactive Display 65-inch", "base_price": 250000, "variance": 0.30},
            {"name": "Projector HD", "base_price": 65000, "variance": 0.25},
            {"name": "Projector 4K", "base_price": 180000, "variance": 0.30},
        ],
        "vendors": ["Samsung", "LG", "Dell", "BenQ", "Epson", "Sony"]
    },
    "Communication Systems": {
        "items": [
            {"name": "IP Phone System", "base_price": 25000, "variance": 0.20},
            {"name": "PBX System", "base_price": 350000, "variance": 0.30},
            {"name": "Video Conferencing System", "base_price": 150000, "variance": 0.25},
            {"name": "HF Radio Equipment", "base_price": 85000, "variance": 0.35},
            {"name": "VHF Radio Equipment", "base_price": 45000, "variance": 0.30},
            {"name": "Software Defined Radio", "base_price": 120000, "variance": 0.40},
        ],
        "vendors": ["Cisco", "Avaya", "Polycom", "Yealink", "Motorola", "Icom"]
    },
    "Power Systems": {
        "items": [
            {"name": "UPS 1KVA", "base_price": 12000, "variance": 0.15},
            {"name": "UPS 3KVA", "base_price": 28000, "variance": 0.20},
            {"name": "UPS 10KVA", "base_price": 85000, "variance": 0.25},
            {"name": "Diesel Generator 50KVA", "base_price": 450000, "variance": 0.20},
            {"name": "Solar Panel 300W", "base_price": 12000, "variance": 0.25},
            {"name": "Solar Inverter 5KVA", "base_price": 35000, "variance": 0.30},
        ],
        "vendors": ["APC", "Eaton", "Vertiv", "Microtek", "Sukam", "Luminous"]
    },
    "Security Systems": {
        "items": [
            {"name": "CCTV Camera Indoor", "base_price": 4500, "variance": 0.20},
            {"name": "CCTV Camera Outdoor", "base_price": 7500, "variance": 0.25},
            {"name": "NVR 16-Channel", "base_price": 35000, "variance": 0.20},
            {"name": "Biometric Access Control", "base_price": 65000, "variance": 0.30},
            {"name": "Fire Alarm System", "base_price": 120000, "variance": 0.25},
            {"name": "Intrusion Detection System", "base_price": 45000, "variance": 0.30},
        ],
        "vendors": ["Hikvision", "Dahua", "Axis", "Bosch", "Honeywell", "CP Plus"]
    },
    "Office Furniture": {
        "items": [
            {"name": "Office Desk Executive", "base_price": 15000, "variance": 0.25},
            {"name": "Office Chair Ergonomic", "base_price": 18000, "variance": 0.30},
            {"name": "Filing Cabinet 4-Drawer", "base_price": 12000, "variance": 0.20},
            {"name": "Conference Table 12-Seater", "base_price": 65000, "variance": 0.30},
            {"name": "Modular Workstation", "base_price": 85000, "variance": 0.25},
            {"name": "Bookshelf 5-Tier", "base_price": 8000, "variance": 0.15},
        ],
        "vendors": ["Godrej", "Featherlite", "Ergo", "Fantasy", "Style Spa", "Wipro"]
    },
    "Electrical Equipment": {
        "items": [
            {"name": "Split AC 1.5 Ton", "base_price": 42000, "variance": 0.15},
            {"name": "Split AC 2 Ton", "base_price": 55000, "variance": 0.15},
            {"name": "Ceiling Fan", "base_price": 3500, "variance": 0.20},
            {"name": "Tube Light LED 4ft", "base_price": 600, "variance": 0.15},
            {"name": "MCB Distribution Board", "base_price": 4500, "variance": 0.15},
            {"name": "Stabilizer 5KVA", "base_price": 8000, "variance": 0.20},
        ],
        "vendors": ["LG", "Daikin", "Voltas", "Crompton", "Philips", "Havells"]
    },
    "Audio/Video Equipment": {
        "items": [
            {"name": "PA System 100W", "base_price": 25000, "variance": 0.25},
            {"name": "Sound System 500W", "base_price": 85000, "variance": 0.30},
            {"name": "Digital Signage Display", "base_price": 55000, "variance": 0.25},
            {"name": "Recording System", "base_price": 120000, "variance": 0.30},
            {"name": "Digital Podium", "base_price": 75000, "variance": 0.30},
            {"name": "Lecture Capture System", "base_price": 180000, "variance": 0.35},
        ],
        "vendors": ["JBL", "Bose", "Yamaha", "Sennheiser", "Samsung", "LG"]
    }
}

SERVICE_CATEGORIES = {
    "IT Services": {
        "items": [
            {"name": "Network Maintenance Annual", "base_price": 250000, "variance": 0.30},
            {"name": "Server Maintenance Annual", "base_price": 180000, "variance": 0.25},
            {"name": "Software AMC", "base_price": 150000, "variance": 0.30},
            {"name": "Cloud Migration Service", "base_price": 500000, "variance": 0.40},
            {"name": "Cybersecurity Audit", "base_price": 300000, "variance": 0.35},
            {"name": "Data Backup Service Monthly", "base_price": 25000, "variance": 0.20},
        ]
    },
    "Facility Management": {
        "items": [
            {"name": "Housekeeping Services Monthly", "base_price": 35000, "variance": 0.20},
            {"name": "Security Guard Services Monthly", "base_price": 80000, "variance": 0.25},
            {"name": "Pest Control Quarterly", "base_price": 8000, "variance": 0.20},
            {"name": "Garden Maintenance Monthly", "base_price": 12000, "variance": 0.25},
            {"name": "Electrical Maintenance Annual", "base_price": 45000, "variance": 0.20},
            {"name": "Building Management System Annual", "base_price": 180000, "variance": 0.30},
        ]
    },
    "Professional Services": {
        "items": [
            {"name": "Web Development Project", "base_price": 200000, "variance": 0.35},
            {"name": "Mobile App Development", "base_price": 400000, "variance": 0.40},
            {"name": "ERP Implementation", "base_price": 800000, "variance": 0.35},
            {"name": "Training Program 5-Day", "base_price": 150000, "variance": 0.30},
            {"name": "Consulting Service Man-Day", "base_price": 25000, "variance": 0.25},
            {"name": "Document Digitization Project", "base_price": 50000, "variance": 0.20},
        ]
    }
}

LOCATIONS = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata", 
             "Hyderabad", "Pune", "Ahmedabad", "Jaipur", "Lucknow",
             "Chandigarh", "Bhopal", "Thiruvananthapuram", "Guwahati", "Patna"]

SOURCES = ["GeM Portal", "Central Public Procurement Portal", "State e-Procurement",
           "Vendor Quotation", "Previous Purchase Order", "Industry Report", 
           "Online Marketplace", "Dealer Price List", "Tender Document"]

VENDOR_RATINGS = {
    "Tier 1": {"price_factor": 1.15, "quality": 0.98},
    "Tier 2": {"price_factor": 1.00, "quality": 0.95},
    "Tier 3": {"price_factor": 0.85, "quality": 0.88},
}

print(f"Product Categories: {len(PRODUCT_CATEGORIES)}")
print(f"Service Categories: {len(SERVICE_CATEGORIES)}")
print(f"Locations: {len(LOCATIONS)}")
print(f"Sources: {len(SOURCES)}")

In [ ]:
def get_season(month):
    """Get demand factor based on month (quarter end = higher demand)"""
    if month <= 3: return 1.0
    elif month <= 6: return 0.95
    elif month <= 9: return 0.90
    else: return 1.10  # Q4 - end of financial year

def generate_products(num_entries=5000):
    """Generate synthetic product pricing dataset"""
    data = []
    for i in range(num_entries):
        category = random.choice(list(PRODUCT_CATEGORIES.keys()))
        cat_data = PRODUCT_CATEGORIES[category]
        item = random.choice(cat_data["items"])
        vendor = random.choice(cat_data["vendors"])
        location = random.choice(LOCATIONS)
        source = random.choice(SOURCES)
        vendor_tier = random.choice(list(VENDOR_RATINGS.keys()))
        
        days_ago = random.randint(0, 730)
        date = datetime.now() - timedelta(days=days_ago)
        demand_factor = get_season(date.month)
        
        # Calculate price with multiple factors
        price = item["base_price"] * (1 + np.random.uniform(-item["variance"], item["variance"]))
        price *= VENDOR_RATINGS[vendor_tier]["price_factor"]
        
        metro_cities = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
        if location in metro_cities:
            price *= np.random.uniform(1.02, 1.08)
        
        price *= np.random.uniform(0.90, 1.10)
        price *= demand_factor
        
        quantity = random.choice([1, 2, 3, 5, 10, 20, 50, 100])
        discount = random.uniform(0.02, 0.08) if quantity >= 10 else random.uniform(0.01, 0.05) if quantity >= 5 else 0
        
        unit_price = round(price * (1 - discount), 2)
        warranty = random.choice([1, 2, 3, 5])
        delivery_days = random.randint(3, 30)
        quality_rating = round(np.random.uniform(3.5, 5.0), 1)
        
        data.append({
            "id": f"PRD-{i+1:06d}", "category": category, "item_name": item["name"],
            "vendor": vendor, "vendor_tier": vendor_tier, "location": location,
            "date": date.strftime("%Y-%m-%d"), "source": source, "quantity": quantity,
            "unit_price": unit_price, "total_price": round(unit_price * quantity, 2),
            "discount_percent": round(discount * 100, 2), "warranty_years": warranty,
            "delivery_days": delivery_days, "quality_rating": quality_rating,
            "base_price": item["base_price"], "demand_factor": demand_factor
        })
    return pd.DataFrame(data)

def generate_services(num_entries=2000):
    """Generate synthetic service pricing dataset"""
    data = []
    for i in range(num_entries):
        category = random.choice(list(SERVICE_CATEGORIES.keys()))
        item = random.choice(SERVICE_CATEGORIES[category]["items"])
        location = random.choice(LOCATIONS)
        source = random.choice(SOURCES)
        
        days_ago = random.randint(0, 730)
        date = datetime.now() - timedelta(days=days_ago)
        demand_factor = get_season(date.month)
        
        provider_rating = random.choice(["A", "B", "C"])
        rating_factor = {"A": 1.10, "B": 1.00, "C": 0.90}[provider_rating]
        
        price = item["base_price"] * (1 + np.random.uniform(-item["variance"], item["variance"]))
        price *= rating_factor * demand_factor
        
        metro_cities = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
        if location in metro_cities:
            price *= np.random.uniform(1.05, 1.15)
        
        duration = random.choice([1, 3, 6, 12])
        
        data.append({
            "id": f"SRV-{i+1:06d}", "category": category, "service_name": item["name"],
            "location": location, "date": date.strftime("%Y-%m-%d"), "source": source,
            "duration_months": duration, "monthly_price": round(price, 2),
            "total_price": round(price * duration, 2), "provider_rating": provider_rating,
            "sla_compliance": round(np.random.uniform(0.85, 0.99), 3),
            "customer_rating": round(np.random.uniform(3.0, 5.0), 1),
            "base_price": item["base_price"], "demand_factor": demand_factor
        })
    return pd.DataFrame(data)

# Generate datasets
print("Generating datasets...")
products_df = generate_products(5000)
services_df = generate_services(2000)

print(f"\nProducts Dataset: {len(products_df)} entries")
print(f"Services Dataset: {len(services_df)} entries")

# Save to CSV
products_df.to_csv("products_prices.csv", index=False)
services_df.to_csv("services_prices.csv", index=False)
print("\nDatasets saved to CSV files!")

## 3. Data Exploration

In [ ]:
# Products dataset overview
print("=" * 60)
print("PRODUCTS DATASET")
print("=" * 60)
print(f"Shape: {products_df.shape}")
print(f"\nColumns: {products_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
products_df.head()

In [ ]:
# Products statistics
print("\nProducts by Category:")
print(products_df['category'].value_counts())

print("\n\nPrice Statistics (INR):")
print(products_df[['unit_price', 'total_price']].describe())

print("\n\nVendor Distribution:")
print(products_df['vendor_tier'].value_counts())

In [ ]:
# Services dataset overview
print("=" * 60)
print("SERVICES DATASET")
print("=" * 60)
print(f"Shape: {services_df.shape}")
print(f"\nColumns: {services_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
services_df.head()

In [ ]:
# Services statistics
print("\nServices by Category:")
print(services_df['category'].value_counts())

print("\n\nPrice Statistics (INR):")
print(services_df[['monthly_price', 'total_price']].describe())

## 4. Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Product prices by category
products_df.groupby('category')['unit_price'].mean().sort_values().plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Average Product Price by Category')
axes[0, 0].set_xlabel('Price (INR)')

# Service prices by category
services_df.groupby('category')['monthly_price'].mean().sort_values().plot(kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Average Service Price by Category')
axes[0, 1].set_xlabel('Monthly Price (INR)')

# Price distribution
axes[1, 0].hist(products_df['unit_price'], bins=30, color='steelblue', edgecolor='black')
axes[1, 0].set_title('Product Price Distribution')
axes[1, 0].set_xlabel('Price (INR)')
axes[1, 0].set_ylabel('Frequency')

# Location-wise average price
products_df.groupby('location')['unit_price'].mean().sort_values().plot(kind='barh', ax=axes[1, 1], color='green')
axes[1, 1].set_title('Average Price by Location')
axes[1, 1].set_xlabel('Price (INR)')

plt.tight_layout()
plt.savefig('data_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved!")

## 5. AI Model Training

In [ ]:
class PriceBenchmarkingModel:
    """
    AI Model for Government Procurement Price Benchmarking.
    Uses feature engineering and ensemble methods for accurate predictions.
    """
    
    def __init__(self):
        self.product_model = None
        self.service_model = None
        self.label_encoders = {}
        self.scalers = {}
        self.product_features = None
        self.service_features = None
        
    def _engineer_product_features(self, df):
        """Create advanced features for product pricing"""
        df = df.copy()
        
        # Category price ratio
        category_avg = df.groupby('category')['base_price'].transform('mean')
        df['category_price_ratio'] = df['base_price'] / category_avg
        
        # Vendor tier factor
        tier_map = {'Tier 1': 1.15, 'Tier 2': 1.0, 'Tier 3': 0.85}
        df['tier_factor'] = df['vendor_tier'].map(tier_map).fillna(1.0)
        
        # Metro city flag
        metro_cities = ['Delhi', 'Mumbai', 'Bangalore', 'Chennai', 'Kolkata', 'Hyderabad', 'Pune']
        df['is_metro'] = df['location'].isin(metro_cities).astype(int)
        
        # Source reliability
        source_map = {'GeM Portal': 0.95, 'Central Public Procurement Portal': 0.93,
                      'State e-Procurement': 0.92, 'Vendor Quotation': 0.88,
                      'Previous Purchase Order': 0.90, 'Industry Report': 0.85,
                      'Online Marketplace': 0.87, 'Dealer Price List': 0.89, 'Tender Document': 0.91}
        df['source_reliability'] = df['source'].map(source_map).fillna(0.85)
        
        # Quantity tier for discount
        df['quantity_tier'] = pd.cut(df['quantity'], bins=[0, 5, 20, 50, 100, 1000],
                                     labels=[0, 0.02, 0.05, 0.08, 0.12]).astype(float)
        
        # Time features
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['is_quarter_end'] = df['month'].isin([3, 6, 9, 12]).astype(int)
        
        # Category statistics
        cat_stats = df.groupby('category')['unit_price'].agg(['mean', 'std', 'min', 'max'])
        df = df.merge(cat_stats, left_on='category', right_index=True, how='left', suffixes=('', '_cat'))
        
        # Encode categoricals
        categorical_cols = ['category', 'vendor_tier', 'location', 'source', 'month', 'quarter']
        for col in categorical_cols:
            if col not in self.label_encoders:
                self.label_encoders[col] = LabelEncoder()
                df[f'{col}_enc'] = self.label_encoders[col].fit_transform(df[col].astype(str))
            else:
                df[f'{col}_enc'] = df[col].map(
                    lambda x: self.label_encoders[col].transform([x])[0]
                    if x in self.label_encoders[col].classes_ else -1
                )
        
        feature_cols = ['base_price', 'quantity', 'warranty_years', 'delivery_days',
                       'demand_factor', 'tier_factor', 'is_metro', 'source_reliability',
                       'quantity_tier', 'is_quarter_end', 'category_price_ratio',
                       'mean', 'std', 'min', 'max', 'category_enc', 'vendor_tier_enc',
                       'location_enc', 'source_enc', 'month_enc', 'quarter_enc']
        
        self.product_features = feature_cols
        return df[feature_cols].fillna(0)
    
    def _engineer_service_features(self, df):
        """Create advanced features for service pricing"""
        df = df.copy()
        
        category_avg = df.groupby('category')['base_price'].transform('mean')
        df['category_price_ratio'] = df['base_price'] / category_avg
        
        metro_cities = ['Delhi', 'Mumbai', 'Bangalore', 'Chennai', 'Kolkata', 'Hyderabad', 'Pune']
        df['is_metro'] = df['location'].isin(metro_cities).astype(int)
        
        df['duration_category'] = pd.cut(df['duration_months'], bins=[0, 3, 6, 12, 24],
                                         labels=['short', 'medium', 'long', 'very_long'])
        
        source_map = {'GeM Portal': 0.95, 'Central Public Procurement Portal': 0.93,
                      'Vendor Quotation': 0.88, 'Tender Document': 0.91, 'Industry Report': 0.85}
        df['source_reliability'] = df['source'].map(source_map).fillna(0.85)
        
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        
        cat_stats = df.groupby('category')['monthly_price'].agg(['mean', 'std'])
        df = df.merge(cat_stats, left_on='category', right_index=True, how='left', suffixes=('', '_cat'))
        
        categorical_cols = ['category', 'location', 'source', 'month', 'quarter', 'duration_category']
        for col in categorical_cols:
            if col not in self.label_encoders:
                self.label_encoders[col] = LabelEncoder()
                df[f'{col}_enc'] = self.label_encoders[col].fit_transform(df[col].astype(str))
            else:
                df[f'{col}_enc'] = df[col].map(
                    lambda x: self.label_encoders[col].transform([x])[0]
                    if x in self.label_encoders[col].classes_ else -1
                )
        
        feature_cols = ['base_price', 'duration_months', 'demand_factor', 'is_metro',
                       'source_reliability', 'category_price_ratio', 'mean', 'std',
                       'category_enc', 'location_enc', 'source_enc', 'month_enc',
                       'quarter_enc', 'duration_category_enc']
        
        self.service_features = feature_cols
        return df[feature_cols].fillna(0)
    
    def train(self, products_df, services_df):
        """Train both product and service models"""
        print("=" * 60)
        print("TRAINING AI MODELS")
        print("=" * 60)
        
        # Train product model
        print("\n1. Training Product Price Model...")
        X_prod = self._engineer_product_features(products_df)
        y_prod = products_df['unit_price'].values
        
        X_train, X_test, y_train, y_test = train_test_split(X_prod, y_prod, test_size=0.2, random_state=42)
        
        self.scalers['product'] = StandardScaler()
        X_train_scaled = self.scalers['product'].fit_transform(X_train)
        X_test_scaled = self.scalers['product'].transform(X_test)
        
        # Train and compare models
        rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1)
        gb = GradientBoostingRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)
        
        rf.fit(X_train_scaled, y_train)
        gb.fit(X_train_scaled, y_train)
        
        rf_score = r2_score(y_test, rf.predict(X_test_scaled))
        gb_score = r2_score(y_test, gb.predict(X_test_scaled))
        
        print(f"  Random Forest R²: {rf_score:.4f}")
        print(f"  Gradient Boosting R²: {gb_score:.4f}")
        
        if rf_score > gb_score:
            self.product_model = rf
            print(f"  -> Selected: Random Forest (R² = {rf_score:.4f})")
        else:
            self.product_model = gb
            print(f"  -> Selected: Gradient Boosting (R² = {gb_score:.4f})")
        
        y_pred = self.product_model.predict(X_test_scaled)
        print(f"  MAE: ₹{mean_absolute_error(y_test, y_pred):,.2f}")
        print(f"  RMSE: ₹{np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")
        
        # Train service model
        print("\n2. Training Service Price Model...")
        X_svc = self._engineer_service_features(services_df)
        y_svc = services_df['monthly_price'].values
        
        X_train, X_test, y_train, y_test = train_test_split(X_svc, y_svc, test_size=0.2, random_state=42)
        
        self.scalers['service'] = StandardScaler()
        X_train_scaled = self.scalers['service'].fit_transform(X_train)
        X_test_scaled = self.scalers['service'].transform(X_test)
        
        self.service_model = GradientBoostingRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)
        self.service_model.fit(X_train_scaled, y_train)
        
        svc_score = r2_score(y_test, self.service_model.predict(X_test_scaled))
        y_pred = self.service_model.predict(X_test_scaled)
        
        print(f"  Gradient Boosting R²: {svc_score:.4f}")
        print(f"  MAE: ₹{mean_absolute_error(y_test, y_pred):,.2f}")
        print(f"  RMSE: ₹{np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")
        
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE!")
        print("=" * 60)
        
        return {
            'product_r2': max(rf_score, gb_score),
            'service_r2': svc_score
        }
    
    def predict_product(self, category, item_name, vendor_tier, location, source,
                       quantity, warranty_years, delivery_days, base_price, demand_factor=1.0):
        """Predict product price"""
        input_data = pd.DataFrame([{
            'category': category, 'item_name': item_name, 'vendor_tier': vendor_tier,
            'location': location, 'source': source, 'quantity': quantity,
            'warranty_years': warranty_years, 'delivery_days': delivery_days,
            'base_price': base_price, 'demand_factor': demand_factor,
            'price_variance': 0.2, 'specifications': '{}',
            'date': pd.Timestamp.now().strftime('%Y-%m-%d'), 'unit_price': 0, 'quality_rating': 4.0
        }])
        
        X = self._engineer_product_features(input_data)
        X_scaled = self.scalers['product'].transform(X)
        predicted_price = max(0, self.product_model.predict(X_scaled)[0])
        
        return {
            'predicted_price': round(predicted_price, 2),
            'quantity': quantity,
            'total_price': round(predicted_price * quantity, 2)
        }
    
    def predict_service(self, category, service_name, location, source,
                       duration_months, base_price, demand_factor=1.0):
        """Predict service price"""
        input_data = pd.DataFrame([{
            'category': category, 'service_name': service_name, 'location': location,
            'source': source, 'duration_months': duration_months, 'base_price': base_price,
            'demand_factor': demand_factor, 'price_variance': 0.2, 'specifications': '{}',
            'date': pd.Timestamp.now().strftime('%Y-%m-%d'), 'monthly_price': 0
        }])
        
        X = self._engineer_service_features(input_data)
        X_scaled = self.scalers['service'].transform(X)
        predicted_price = max(0, self.service_model.predict(X_scaled)[0])
        
        return {
            'predicted_monthly_price': round(predicted_price, 2),
            'duration_months': duration_months,
            'total_price': round(predicted_price * duration_months, 2)
        }
    
    def save_model(self, filepath='price_benchmarking_model.pkl'):
        """Save model to file"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'product_model': self.product_model,
                'service_model': self.service_model,
                'label_encoders': self.label_encoders,
                'scalers': self.scalers,
                'product_features': self.product_features,
                'service_features': self.service_features
            }, f)
        print(f"\nModel saved to {filepath}")
    
    def load_model(self, filepath='price_benchmarking_model.pkl'):
        """Load model from file"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        self.product_model = data['product_model']
        self.service_model = data['service_model']
        self.label_encoders = data['label_encoders']
        self.scalers = data['scalers']
        self.product_features = data['product_features']
        self.service_features = data['service_features']
        print(f"Model loaded from {filepath}")

print("PriceBenchmarkingModel class defined!")

In [ ]:
# Initialize and train the model
model = PriceBenchmarkingModel()
metrics = model.train(products_df, services_df)

## 6. Model Evaluation

In [ ]:
# Feature importance for product model
if hasattr(model.product_model, 'feature_importances_'):
    feature_imp = pd.DataFrame({
        'feature': model.product_features,
        'importance': model.product_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_imp['feature'][:15], feature_imp['importance'][:15], color='steelblue')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title('Top 15 Feature Importance (Product Model)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=100)
    plt.show()
    
    print("\nTop 10 Features:")
    print(feature_imp.head(10).to_string(index=False))

## 7. Price Prediction Examples

In [ ]:
# Get base prices from dataset
router_base = products_df[products_df['item_name'] == 'Enterprise Router']['base_price'].iloc[0]
maint_base = services_df[services_df['service_name'] == 'Network Maintenance Annual']['base_price'].iloc[0]

print("=" * 60)
print("EXAMPLE PREDICTIONS")
print("=" * 60)

# Product prediction
print("\n1. PRODUCT: Enterprise Router")
print("-" * 40)
pred = model.predict_product(
    category="Networking Equipment",
    item_name="Enterprise Router",
    vendor_tier="Tier 1",
    location="Delhi",
    source="GeM Portal",
    quantity=10,
    warranty_years=3,
    delivery_days=14,
    base_price=router_base
)
print(f"  Predicted Unit Price: ₹{pred['predicted_price']:,.2f}")
print(f"  Quantity: {pred['quantity']}")
print(f"  Total Price: ₹{pred['total_price']:,.2f}")

# Service prediction
print("\n2. SERVICE: Network Maintenance Annual")
print("-" * 40)
pred = model.predict_service(
    category="IT Services",
    service_name="Network Maintenance Annual",
    location="Mumbai",
    source="Vendor Quotation",
    duration_months=12,
    base_price=maint_base
)
print(f"  Predicted Monthly Price: ₹{pred['predicted_monthly_price']:,.2f}")
print(f"  Duration: {pred['duration_months']} months")
print(f"  Total Price: ₹{pred['total_price']:,.2f}")

# More examples
print("\n3. MORE EXAMPLES")
print("-" * 40)

examples = [
    {"cat": "Computing Hardware", "item": "Laptop Standard", "base": 55000, "qty": 20},
    {"cat": "Security Systems", "item": "CCTV Camera Outdoor", "base": 7500, "qty": 50},
    {"cat": "Electrical Equipment", "item": "Split AC 1.5 Ton", "base": 42000, "qty": 5},
]

for ex in examples:
    pred = model.predict_product(
        category=ex['cat'], item_name=ex['item'], vendor_tier="Tier 2",
        location="Bangalore", source="GeM Portal", quantity=ex['qty'],
        warranty_years=2, delivery_days=10, base_price=ex['base']
    )
    print(f"\n  {ex['item']} (Qty: {ex['qty']}):")
    print(f"    Unit Price: ₹{pred['predicted_price']:,.2f}")
    print(f"    Total: ₹{pred['total_price']:,.2f}")

## 8. Save Model

In [ ]:
# Save the trained model
model.save_model('price_benchmarking_model.pkl')

# Verify by loading
print("\nVerifying model save/load...")
loaded_model = PriceBenchmarkingModel()
loaded_model.load_model('price_benchmarking_model.pkl')

# Test prediction with loaded model
pred = loaded_model.predict_product(
    category="Networking Equipment",
    item_name="Enterprise Router",
    vendor_tier="Tier 1",
    location="Delhi",
    source="GeM Portal",
    quantity=10,
    warranty_years=3,
    delivery_days=14,
    base_price=router_base
)
print(f"\nLoaded Model Test Prediction: ₹{pred['predicted_price']:,.2f}")
print("\n" + "=" * 60)
print("ALL FILES GENERATED:")
print("=" * 60)
print("1. products_prices.csv - Product pricing dataset (5000 entries)")
print("2. services_prices.csv - Service pricing dataset (2000 entries)")
print("3. price_benchmarking_model.pkl - Trained AI model")
print("4. data_visualization.png - Data visualization charts")
print("5. feature_importance.png - Model feature importance")